[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sapiosciences/sapio-py-tutorials/blob/master/02_query_data_records.ipynb)

# Create Connection

Every notebook starts by creating a *SapioUser*. That object holds the webservice URL and the credentials for the request.

How to authenticate is covered in [API Authentication](01_api_authentication.ipynb). An API user from User Manager signs in with a username and password. An API key from the Webservice API page grants admin access. A bearer token can be used when the system has OAuth configured. The connection below uses an API user.

`verify_ssl_cert` is false here because the example server uses a self-signed certificate. Leave it true when the server has a real certificate.

In [ ]:
from sapiopylib.rest.User import SapioUser
from sapiopylib.rest.DataRecordManagerService import DataRecord
from sapiopylib.rest.DataRecordManagerService import DataRecordPojoPageCriteria
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.DataRecordManagerService import DataRecordPojoHierarchyPageCriteria

user = SapioUser(url="https://linux-vm:8443/webservice/api",
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7", account_name="sapio",
                 username="pyRestTest", password="Password1!", verify_ssl_cert=False)
dataRecordManager = DataMgmtServer.get_data_record_manager(user)

These commands do not produce any results yet.

Now let's try to query for record ID = 1

# Query Data Records

In [ ]:
# Get all Directory records whose ids are in the list [1] (so just a single record with id = 1)
result = dataRecordManager.query_data_records_by_id("Directory", [1])
if len(result.result_list) > 0:
    result_list: list = result.result_list
    element: DataRecord = result_list[0]
    print(element.get_record_id())
    print(element.get_fields())

In example above, we query a directory whose record ID is 1 (which by the way is the root directory. We print the record ID and the fields of the record.
Alternatively, you can direct print a DataRecordPojo object or a page's result directly. It will show the data record name of these data records.

In [ ]:
# Get all Samples with SampleId in the list ["000007"]
result = dataRecordManager.query_data_records("Sample", "SampleId", ["000007"])
print(result)

Paging is available. In the next example we are going to get samples, 10 samples per page for 3 pages.

In [ ]:
next_page = DataRecordPojoPageCriteria(page_size=10)
for i in range(3):
    result = dataRecordManager.query_all_records_of_type("Sample", next_page)
    next_page = result.next_page_criteria
    print("Page " + str(i + 1) + ": " + str(result))
    if not result.is_next_page_available:
        break

query_system_for_record returns either the DataRecordPojo when it finds a record, or a "None" object if it's not found.

In [ ]:
result = dataRecordManager.query_system_for_record('Directory', 1)
print(result)
result = dataRecordManager.query_system_for_record('Directory', 1000000)
print(result)

You can get parents and children of the records.
The easiest form is to take a single record and get children of that.

In this example, we find one FCS file and try to get its parent sample.

In [ ]:
fcs_result = dataRecordManager.query_all_records_of_type("FCSFile", DataRecordPojoPageCriteria(page_size=1))
if len(fcs_result.result_list) > 0:
    fcs_record = fcs_result.result_list[0]
    parents = dataRecordManager.get_parents(fcs_record.get_record_id(), "FCSFile", "Sample")
    if len(parents.result_list) > 0:
        parent_sample = parents.result_list[0]
        print(parent_sample)

In a slightly more complex case, we will be using a batch call to retrieve parents of multiple FCS children at once.

It is often much faster to use batch call than calling records in a loop. This will reduce the HTTP and remote overhead.

The result object is slightly different. To demonstrate paging in this situation, we will try to have a page of 10 on parents, for first five pages of parents for first 100 FCS records.
Note: Since each FCS record has a parent, this will not complete the entire query. The remaining pages will simply never be computed nor returned here.

In [ ]:
fcs_result = dataRecordManager.query_all_records_of_type("FCSFile", DataRecordPojoPageCriteria(page_size=100))
samples = list()
if len(fcs_result.result_list) > 0:
    fcs_record_id_list = [x.get_record_id() for x in fcs_result.result_list]

    fcs_record_by_record_id = dict((x.get_record_id(), x) for x in fcs_result.result_list)

    next_parent_page_criteria = DataRecordPojoHierarchyPageCriteria(page_size=10)
    for i in range(5):
        print("Page " + str(i + 1))

        parents = dataRecordManager.get_parents_list(fcs_record_id_list, "FCSFile", "Sample", next_parent_page_criteria)
        next_parent_page_criteria = parents.next_page_criteria

        for source_record_id, result_records in parents.result_map.items():
            if result_records is None or len(result_records) == 0:
                continue
            
            samples.extend(result_records)

            fcs_record = fcs_record_by_record_id.get(source_record_id)
            if fcs_record is not None:
                print(str(fcs_record) + " -> [" + ', '.join([str(x) for x in result_records]) + "]")
                
        if not parents.is_next_page_available:
            break

You can translate data in a list of data records into a pandas data frame, which can be used by external tools!

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 100)
samples_data_frame = dataRecordManager.get_data_frame(samples)
display(samples_data_frame)

# Auto-Paging

Recall in the previous sections of the tutorial we had a paging loop like this:


```
next_page = DataRecordPojoPageCriteria(page_size=10)
for i in range(3):
    result = dataRecordManager.query_all_records_of_type("Sample", next_page)
    next_page = result.next_page_criteria
    print("Page " + str(i + 1) + ": " + str(result))
    if not result.is_next_page_available:
        break
```

You may have to page through the records in the similar fashion all time. You may not really be concerned about which page it is in, but rather just be able to get all records (or all records up to a limit determined by you).

In this case, using our auto-paging API can save quite a bit of work:

In [ ]:
from sapiopylib.rest.utils.autopaging import *

pager = QueryAllRecordsOfTypeAutoPager("Directory", user, DataRecordPojoPageCriteria(page_size=10))
for record in pager:
  print(str(record))

In the example above, we see that all auto-paging classes from sapiopylib can be imported through sapiopylib.rest.utils.autopaging module.
We also see that each auto-pager class implements its own iterator. Each auto-pager acts as one-time iterator by itself (with no ability of going backward. If you want to revisit elements, you may need to store records you have obtained in another container such as List or a file)

It is also possible to obtain all results in a single call. For classes such as QueryAllRecordsOfTypeAutoPager it may be easy to understand the differences between iterating in a loop and obtaining all results at once. All we would have been doing is basically collecting them results into a single list.
So the code above would have the following effect as the code below

In [ ]:
pager = QueryAllRecordsOfTypeAutoPager("Directory", user, DataRecordPojoPageCriteria(page_size=3))
all_records: List[DataRecord] = pager.get_all_at_once()

for record in all_records:
  print(str(record))

In other pagers such as hierarchical pagers and side link pagers, there are subtle differences between the two ways you may have to remember.

Consider the following code

In [ ]:
children_pager = GetChildrenListAutoPager([x.record_id for x in all_records], "Directory",
                                                  user, DataRecordPojoHierarchyPageCriteria(page_size=3))

The children_pager variable is also an iterator. But this time, it is not an iterator of data records. Rather it is a **partial** dictionary of (record ID) -> children records (of data type "Directory", in this case)
What does that mean for it to be a partial dictionary? It means that:


1.   Each occurance of the record ID key does not completely visit all of its children.
2.   The entire children list for each record ID of provided record ids is equal to the union of all results keyed by the same record ID.

This signature is the same signature that is returend by dataRecordManager and that of the Sapio REST APi specfication. A partial result can be returned in such as case where if any of the record has x number of children, where x does not divide the page size (with no remainders).


Therefore, in order to properly collect the all record IDs for all provided records, we must combine the union of all such sets of results:

In [ ]:
# A Multimap can have the same key multiple times with different values
# In the code below all_results is a SetMultimap which maps ids (int) to sets of child DataRecords
# It may be thought as a dict that looks like 
# {
#   1: {DataRecord 1, DataRecord 2},
#   2: {DataRecord 3, DataRecord 4},
#   1: {DataRecord 5, DataRecord 6}
# }
# 
# The second entry with 1 as the key doesn't replace the first
all_results: SetMultimap[int, DataRecord] = SetMultimap()
for parent_record_id, children_records_partial in children_pager:
  all_results.put_all(parent_record_id, children_records_partial)

It would be **incorrect** if we were to replace the key in dictionary for each parent_record_id, as that would eliminate the partial results entered the earlier page.

However, you can save all those work collecting the partial results, if you use the other method to obtain all records at once. Assuming the entire result can fit into the heap of your runtime memory.

Therefore, the code above is equivalent to:

In [ ]:
all_results: SetMultimap[int, DataRecord] = children_pager.get_all_at_once()

Note that if you are trying to use record model then implicitly it would already mean you made the decision to fit all pages into one heap of the memory.